In [ ]:
# Cell 1: Environment & Repository Setup
import os
import sys
import subprocess
from pathlib import Path

print("--- Setting Up Humanoid-Face-3D Workspace ---")
target_dir = Path("/kaggle/working/Humanoid-Face-3D")
if not (target_dir / "src/pipeline.py").exists():
    subprocess.run(["git", "clone", "--recurse-submodules", "https://github.com/NetPranav/Humanoid-Face-3D.git", str(target_dir)], check=True)
os.chdir(str(target_dir))
subprocess.run(["git", "fetch", "origin", "main"], check=False)
subprocess.run(["git", "reset", "--hard", "origin/main"], check=False)
subprocess.run(["git", "submodule", "update", "--init", "--recursive"], check=False)

if str(target_dir) not in sys.path:
    sys.path.insert(0, str(target_dir))
if "." not in sys.path:
    sys.path.insert(0, ".")
print("Working directory:", os.getcwd())


In [ ]:
# Cell 2: Install GPU Dependencies
import subprocess
subprocess.run(["pip", "install", "insightface", "onnxruntime-gpu", "trimesh", "pyyaml", "scipy", "Pillow", "opencv-python", "--quiet"], check=True)

import torch
print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({props.total_memory / 1e9:.1f} GB VRAM)")


In [ ]:
# Cell 3: Install Portable Headless Blender 4.1.0 into /tmp (Zero Quota)
import os
import subprocess
from pathlib import Path

print("--- Installing Portable Headless Blender 4.1.0 into /tmp ---")
blender_bin = Path("/tmp/blender/blender")
if not blender_bin.exists():
    sh_script = Path("scripts/install_portable_blender.sh")
    if sh_script.exists():
        subprocess.run(["bash", str(sh_script)], check=True)
    else:
        # Direct download fallback
        subprocess.run(["mkdir", "-p", "/tmp/blender"], check=True)
        subprocess.run(["wget", "-q", "-O", "/tmp/blender.tar.xz", "https://download.blender.org/release/Blender4.1/blender-4.1.0-linux-x64.tar.xz"], check=True)
        subprocess.run(["tar", "-xf", "/tmp/blender.tar.xz", "-C", "/tmp/blender", "--strip-components=1"], check=True)
        subprocess.run(["rm", "-f", "/tmp/blender.tar.xz"], check=True)

os.environ["PATH"] = f"/tmp/blender:{os.environ.get('PATH', '')}"
blender_ver = subprocess.run(["blender", "--version"], capture_output=True, text=True)
print("Blender Version:", blender_ver.stdout.splitlines()[0] if blender_ver.stdout else "Unknown")


In [ ]:
# Cell 4: Discover & Mount Models (FLAME, MICA, Stage 3 Detail GAN)
import shutil
import subprocess
import tarfile
import json
from pathlib import Path

print("--- Locating & Mounting Model Weights ---")
# 1. FLAME 2020 Model
flame_pkl_candidates = list(Path("/kaggle/input").glob("**/generic_model.pkl")) + list(Path(".").glob("**/generic_model.pkl"))
if not flame_pkl_candidates:
    raise FileNotFoundError("generic_model.pkl not found! Please attach dataset nightshowdown/flame-model.")
flame_pkl = flame_pkl_candidates[0]
print(f"  Found FLAME Model: {flame_pkl}")

Path("data/flame_model").mkdir(parents=True, exist_ok=True)
if not Path("data/flame_model/generic_model.pkl").exists():
    shutil.copy(flame_pkl, "data/flame_model/generic_model.pkl")

head_candidates = list(Path("/kaggle/input").glob("**/head_template.obj")) + list(Path(".").glob("**/head_template.obj"))
if head_candidates and not Path("data/flame_model/head_template.obj").exists():
    shutil.copy(head_candidates[0], "data/flame_model/head_template.obj")

# 2. MICA Pretrained Weights
Path("models_cache/mica").mkdir(parents=True, exist_ok=True)
dest_tar = Path("models_cache/mica/pretrained.tar")
if not dest_tar.exists() or dest_tar.stat().st_size < 100_000_000:
    mica_candidates = list(Path("/kaggle/input").glob("**/pretrained.tar")) + list(Path("/kaggle/input").glob("**/mica.tar"))
    if mica_candidates:
        print(f"  Mounting MICA from: {mica_candidates[0]}")
        shutil.copy(mica_candidates[0], dest_tar)
    else:
        print("  Streaming MICA from Google Drive CDN (~5s)...")
        subprocess.run([
            "curl", "-L", "--progress-bar",
            "https://drive.usercontent.google.com/download?id=1bYsI_spptzyuFmfLYqYkcJA6GZWZViNt&export=download&confirm=t",
            "-o", str(dest_tar)
        ], check=True)

# 3. Stage 3 Detail GAN EMA Generator & Stats
Path("models_cache/stage3_detail").mkdir(parents=True, exist_ok=True)
ema_target = Path("models_cache/stage3_detail/ema_generator.pt")
stats_target = Path("models_cache/stage3_detail/normalization_stats.json")

# Check if detail_gan_assets.tar.gz was mounted from phase-3-detail-gan-train
gan_archives = list(Path("/kaggle/input").glob("**/detail_gan_assets.tar.gz"))
if gan_archives:
    print(f"  Found Detail GAN archive: {gan_archives[0]}, extracting...")
    try:
        with tarfile.open(gan_archives[0], "r:gz") as tar:
            tar.extractall("models_cache/stage3_detail")
    except Exception as e:
        print(f"  Warning extracting archive: {e}")

if not ema_target.exists():
    gan_candidates = list(Path("/kaggle/input").glob("**/ema_generator.pt")) + list(Path(".").glob("**/ema_generator.pt"))
    if gan_candidates:
        print(f"  Mounting Detail GAN from: {gan_candidates[0]}")
        shutil.copy(gan_candidates[0], ema_target)
    else:
        print("  Detail GAN not in /kaggle/input; looking in repository checkpoints...")
        ckpt_local = Path("checkpoints/stage3_detail/ema_generator.pt")
        if ckpt_local.exists():
            shutil.copy(ckpt_local, ema_target)

if not stats_target.exists():
    stats_candidates = list(Path("/kaggle/input").glob("**/normalization_stats.json")) + list(Path(".").glob("**/normalization_stats.json"))
    if stats_candidates:
        shutil.copy(stats_candidates[0], stats_target)
    else:
        with open(stats_target, "w") as f:
            json.dump({"p99_displacement_mm": 1.1465, "p99_mm": 1.1465, "resolution": 1024}, f)

print(f"All core models mounted: FLAME + MICA + Detail GAN ({ema_target.exists()}).")


In [ ]:
# Cell 5: Run Production End-to-End Pipeline Across All 4 Subjects
import time
import json
import shutil
import subprocess
from pathlib import Path
from src.pipeline import FaceGeoPipeline

subjects = {
    "carell": {
        "hair_preset": "heavy_stubble",
        "stylize": "chiseled",
        "cdn": "https://raw.githubusercontent.com/Zielon/MICA/main/demo/input/carell.jpg",
        "filename": "carell.jpg"
    },
    "connelly": {
        "hair_preset": "clean_shaven",
        "stylize": "neutral",
        "cdn": "https://raw.githubusercontent.com/Zielon/MICA/main/demo/input/connelly.jpg",
        "filename": "connelly.jpg"
    },
    "justin": {
        "hair_preset": "goatee",
        "stylize": "heroic",
        "cdn": "https://raw.githubusercontent.com/Zielon/MICA/main/demo/input/justin.png",
        "filename": "justin.png"
    },
    "lawrence": {
        "hair_preset": "clean_shaven",
        "stylize": "neutral",
        "cdn": "https://raw.githubusercontent.com/Zielon/MICA/main/demo/input/lawrence.jpg",
        "filename": "lawrence.jpg"
    },
}

test_bench_dir = Path("data/benchmark_test")
test_bench_dir.mkdir(parents=True, exist_ok=True)

# Prepare portraits with local fallback or CDN fetch
for s_name, s_info in subjects.items():
    s_dir = test_bench_dir / s_name
    s_dir.mkdir(parents=True, exist_ok=True)
    target_file = s_dir / s_info["filename"]
    if not target_file.exists():
        vendor_cand = Path("vendor/MICA/demo/input") / s_info["filename"]
        if vendor_cand.exists():
            shutil.copy(vendor_cand, target_file)
        else:
            print(f"Downloading portrait for {s_name} from CDN...")
            subprocess.run(["curl", "-L", "-s", s_info["cdn"], "-o", str(target_file)], check=True)

production_base = Path("/kaggle/working/outputs/production_batch")
production_base.mkdir(parents=True, exist_ok=True)

production_results = {}
total_start = time.time()

for subj_name, s_info in subjects.items():
    print(f"\n{'='*70}")
    print(f"🚀 Processing Production Subject: {subj_name.upper()}")
    print(f"   Facial Hair: {s_info['hair_preset']} | Stylization: {s_info['stylize']}")
    print(f"{'='*70}")
    
    subj_out = production_base / subj_name
    subj_out.mkdir(parents=True, exist_ok=True)
    
    # Configure production settings
    pipe_cfg = {
        "pipeline": {"input_min_photos": 1},
        "stage0": {"min_face_confidence": 0.4},
        "stage1": {"enable_dense_fitting": False},
        "stage1_5": {"enabled": True},
        "stage3": {
            "enabled": True,
            "resolution": 1024,
            "checkpoint": "models_cache/stage3_detail/ema_generator.pt",
            "stats": "models_cache/stage3_detail/normalization_stats.json"
        },
        "stage4": {
            "enabled": True,
            "preset": s_info["hair_preset"],
            "resolution": 1024,
            "neck_collar_threshold": 0.20
        },
        "stage5": {
            "stylize": s_info["stylize"],
            "enable_lods": True,
            "enable_armature": True
        },
        "allow_degraded": False
    }
    
    pipeline = FaceGeoPipeline(cfg=pipe_cfg, model_dir="models_cache")
    photos = sorted([str(p) for p in (test_bench_dir / subj_name).glob("*.*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]])
    t0 = time.time()
    res = pipeline.run(photo_paths=photos, output_dir=str(subj_out))
    elapsed = time.time() - t0
    
    print(f"✅ {subj_name.upper()} completed in {elapsed:.1f}s.")
    fbx_display = Path(res['fbx_path']).name if res.get('fbx_path') else 'None'
    print(f"   FBX: {fbx_display}")
    print(f"   Manifest: {Path(res['manifest_path']).name}")
    production_results[subj_name] = res

total_elapsed = time.time() - total_start
print(f"\n{'='*70}")
print(f"🎉 Production Batch Complete! All 4 subjects processed in {total_elapsed:.1f}s.")
print(f"{'='*70}")


In [ ]:
# Cell 6: Verify Unreal Engine 5 FBX Assets & Rigs
import json
from pathlib import Path

production_base = Path("/kaggle/working/outputs/production_batch")

print("--- Unreal Engine 5 Production Asset Audit ---")
for subj_dir in sorted(production_base.iterdir()):
    if not subj_dir.is_dir():
        continue
    
    name = subj_dir.name
    print(f"\nSubject: {name.upper()}")
    
    # 1. Base files
    obj_p = subj_dir / "head_mesh.obj"
    fbx_p = subj_dir / "head_mesh_ue5_livelink.fbx"
    bs_p = subj_dir / "blendshapes_arkit52.json"
    arm_p = subj_dir / "armature_rig.json"
    
    print(f"  • Base OBJ: {'✅' if obj_p.exists() else '❌'} ({obj_p.stat().st_size / 1e3:.1f} KB)")
    print(f"  • UE5 FBX: {'✅' if fbx_p.exists() else '⚠️ (Script Generated)'} ({fbx_p.stat().st_size / 1e6:.2f} MB)" if fbx_p.exists() else "  • UE5 FBX: Pending Blender Execution")
    print(f"  • ARKit-52 Blendshapes: {'✅' if bs_p.exists() else '❌'}")
    print(f"  • 5-Joint Armature Rig: {'✅' if arm_p.exists() else '❌'}")
    
    # 2. Textures
    norm_p = subj_dir / "head_normal_map.png"
    disp_p = subj_dir / "head_displacement_16bit.png"
    print(f"  • 1024² Normal Map: {'✅' if norm_p.exists() else '❌'}")
    print(f"  • 1024² 16-bit Disp: {'✅' if disp_p.exists() else '❌'}")
    
    # 3. Facial Hair
    hair_dir = subj_dir / "facial_hair"
    if hair_dir.exists():
        cards_obj = hair_dir / "facial_hair_cards.obj"
        hair_alpha = hair_dir / "hair_card_alpha.png"
        hair_norm = hair_dir / "hair_card_normal.png"
        print(f"  • Hair Cards Mesh: {'✅' if cards_obj.exists() else 'None'}")
        print(f"  • Hair Card Alpha & Normal: {'✅' if hair_alpha.exists() and hair_norm.exists() else 'None'}")
    
    # 4. LOD Chain
    lod_dir = subj_dir / "lod"
    if lod_dir.exists():
        lods = list(lod_dir.glob("*.obj")) + list(lod_dir.glob("*.fbx"))
        print(f"  • Multi-Tier LODs: Found {len(lods)} LOD asset files")


In [ ]:
# Cell 7: Render Multi-Subject Production Showcase Gallery
import cv2
import numpy as np
from pathlib import Path

production_base = Path("/kaggle/working/outputs/production_batch")
subjects = ["carell", "connelly", "justin", "lawrence"]

tiles = []
for s_name in subjects:
    s_dir = production_base / s_name
    preview_p = s_dir / "preview_render.png"
    if not preview_p.exists():
        preview_p = s_dir / "head_mesh.png"
    norm_p = s_dir / "head_normal_map.png"
    
    # Load or generate preview image
    if preview_p.exists():
        img_preview = cv2.imread(str(preview_p))
        img_preview = cv2.resize(img_preview, (384, 384))
    else:
        img_preview = np.zeros((384, 384, 3), dtype=np.uint8)
        cv2.putText(img_preview, s_name.upper(), (40, 200), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 2)
        
    if norm_p.exists():
        img_norm = cv2.imread(str(norm_p))
        img_norm = cv2.resize(img_norm, (384, 384))
    else:
        img_norm = np.full((384, 384, 3), (255, 128, 128), dtype=np.uint8)
        
    # Stack vertically: Preview on top, Normal map below
    subj_column = np.vstack([img_preview, img_norm])
    
    # Add subject header label
    header = np.zeros((48, 384, 3), dtype=np.uint8)
    cv2.putText(header, s_name.upper(), (20, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 200), 2)
    subj_tile = np.vstack([header, subj_column])
    tiles.append(subj_tile)

# Concatenate all subjects horizontally
gallery = np.hstack(tiles)
out_gallery_p = production_base / "production_batch_showcase.png"
cv2.imwrite(str(out_gallery_p), gallery)
print(f"✅ Production showcase gallery saved to: {out_gallery_p.name} ({gallery.shape[1]}x{gallery.shape[0]})")


In [ ]:
# Cell 8: Package Deliverable Archive & Quota Guard
import shutil
import subprocess
from pathlib import Path

print("--- Packaging Production Bundle ---")
production_base = Path("/kaggle/working/outputs/production_batch")
archive_path = Path("/kaggle/working/outputs_production_batch.tar.gz")

# Compress output directory
subprocess.run(["tar", "-czf", str(archive_path), "-C", "/kaggle/working/outputs", "production_batch"], check=True)
size_mb = archive_path.stat().st_size / 1e6
print(f"✅ Production asset bundle created: {archive_path.name} ({size_mb:.2f} MB)")

# Verify disk usage adheres to Kaggle Quota Guard (< 19.5 GB)
df_res = subprocess.run(["df", "-h", "/kaggle/working"], capture_output=True, text=True)
print("\nDisk Usage Status:\n" + df_res.stdout)
print("Production Cloud Batch Run Complete!")
